# Multi-Model Heatmap — TGPY / Grenada

Reads `canonical_sfc.csv` from each model's latest run and produces a
parameter × time heatmap for all 5 models side-by-side.

**Run order:** run each model's `analysis.ipynb` first, then this notebook.

| Model | Type | Canonical CSV |
|-------|------|---------------|
| ECMWF IFS HRES | Physics deterministic | `../ecmwf_ifs/data/{run}/canonical_sfc.csv` |
| ECMWF AIFS-single | AI deterministic | `../ecmwf_aifs/data/{run}/canonical_sfc.csv` |
| NOAA GFS | Physics deterministic | `../noaa_gfs/data/{run}/canonical_sfc.csv` |
| NOAA GEFS mean | Physics ensemble mean | `../noaa_gefs/data/{run}/canonical_sfc.csv` |
| ECMWF AIFS-ENS mean | AI ensemble mean | `../ecmwf_aifs_ens/data/{run}/canonical_sfc.csv` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Canonical parameter definitions: label, unit, colormap ───────────────────
PARAMS = {
    # Core surface
    "t2m_c":   ("2m Temp",       "°C",    "RdBu_r"),
    "d2m_c":   ("Dewpoint",      "°C",    "RdBu"),
    "msl_hpa": ("MSL Pressure",  "hPa",   "RdBu_r"),
    "sp_hpa":  ("Sfc Pressure",  "hPa",   "RdBu_r"),
    "wspd_kt": ("Wind Speed",    "kt",    "YlOrRd"),
    "fg10_kt": ("Wind Gust",     "kt",    "YlOrRd"),
    "wdir":    ("Wind Dir",      "°",     "twilight"),
    "tp_mm":   ("Precip",        "mm/6h", "Blues"),
    "tcc_pct": ("Cloud Cover",   "%",     "Greys"),
    "tcwv":    ("TCWV",          "kg/m²", "YlGnBu"),
    "cape":    ("CAPE",          "J/kg",  "hot_r"),
    # New tropical params
    "skt_c":   ("Skin Temp",     "°C",    "RdBu_r"),
    "cp_mm":   ("Conv Precip",   "mm/6h", "Blues"),
    "lcc_pct": ("Low Cloud",     "%",     "Greys"),
    "hcc_pct": ("High Cloud",    "%",     "Greys_r"),
    "cin":     ("CIN",           "J/kg",  "PuBu_r"),
    "olr_wm2": ("OLR",           "W/m²",  "hot"),
    "blh_m":   ("BL Height",     "m",     "YlOrBr"),
    "lhf":     ("Latent HF",     "W/m²",  "PuRd"),
    "shf":     ("Sensible HF",   "W/m²",  "OrRd"),
}

# ── Model folders (relative to this notebook) ─────────────────────────────────
MODELS = {
    "ecmwf_ifs":      "IFS HRES",
    "ecmwf_aifs":     "AIFS-single",
    "noaa_gfs":       "NOAA GFS",
    "noaa_gefs":      "GEFS mean",
    "ecmwf_aifs_ens": "AIFS-ENS mean",
}

STEPS_6H = list(range(0, 121, 6))   # 21 steps: T+0 … T+120h
BASE     = Path("..")

print("Setup complete.")
print(f"Parameters : {len(PARAMS)} total  ({list(PARAMS.keys())})")
print(f"Models     : {list(MODELS.keys())}")

In [ ]:
# ── Load canonical CSVs — latest run per model ────────────────────────────────
dfs        = {}   # model_id → DataFrame
run_labels = {}   # model_id → run label string

for model_id in MODELS:
    data_root = BASE / model_id / "data"
    if not data_root.exists():
        print(f"  {model_id:20s} — data/ folder missing, skip")
        continue
    candidates = sorted(
        [d for d in data_root.iterdir()
         if d.is_dir() and (d / "canonical_sfc.csv").exists()],
        reverse=True,
    )
    if not candidates:
        print(f"  {model_id:20s} — no canonical_sfc.csv found (run analysis.ipynb first)")
        continue
    run_dir = candidates[0]
    df = pd.read_csv(run_dir / "canonical_sfc.csv",
                     index_col="valid_time", parse_dates=True)
    df["step_h"] = ((df.index - df.index[0]).total_seconds() / 3600).round().astype(int)
    dfs[model_id]        = df
    run_labels[model_id] = run_dir.name
    print(f"  {model_id:20s} — {run_dir.name}  ({len(df)} steps)")

print(f"\nLoaded {len(dfs)} / {len(MODELS)} models")

In [ ]:
# ── Build heatmap arrays: shape (n_models, n_steps) per parameter ─────────────
model_ids    = list(dfs.keys())
model_labels = [MODELS[m] for m in model_ids]
step_labels  = [f"T+{s:03d}" for s in STEPS_6H]
n_m, n_s     = len(model_ids), len(STEPS_6H)

data = {}
for pid in PARAMS:
    arr = np.full((n_m, n_s), np.nan)
    for mi, mid in enumerate(model_ids):
        df = dfs[mid]
        if pid not in df.columns:
            continue
        for si, step in enumerate(STEPS_6H):
            rows = df[df["step_h"] == step]
            if not rows.empty:
                v = float(rows[pid].iloc[0])
                if not np.isnan(v):
                    arr[mi, si] = v
    data[pid] = arr

# ── Plot ─────────────────────────────────────────────────────────────────────
N_P  = len(PARAMS)
fig, axes = plt.subplots(N_P, 1, figsize=(22, 3.0 * N_P), constrained_layout=True)
if N_P == 1:
    axes = [axes]

for ax, pid in zip(axes, PARAMS):
    label, unit, cmap_name = PARAMS[pid]
    arr = data[pid]

    masked = np.ma.masked_invalid(arr)
    cmap   = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("#dddddd")

    all_vals = arr[~np.isnan(arr)]
    if len(all_vals) == 0:
        vmin, vmax = 0, 1
    else:
        vmin, vmax = all_vals.min(), all_vals.max()
        if vmin == vmax:
            vmin -= 0.5; vmax += 0.5

    im = ax.imshow(masked, aspect="auto", cmap=cmap,
                   vmin=vmin, vmax=vmax, interpolation="nearest")

    ax.set_yticks(range(n_m))
    ax.set_yticklabels(model_labels, fontsize=9)
    ax.set_xticks(range(n_s))
    ax.set_xticklabels(
        [step_labels[i] if i % 4 == 0 else "" for i in range(n_s)],
        fontsize=8, rotation=45, ha="right",
    )
    ax.set_title(f"{label}  ({unit})", fontsize=10, fontweight="bold", loc="left", pad=4)

    # Annotate non-NaN cells
    span = vmax - vmin
    for mi in range(n_m):
        for si in range(n_s):
            v = arr[mi, si]
            if np.isnan(v):
                continue
            txt = f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}"
            brightness = (v - vmin) / span if span > 0 else 0.5
            color = "white" if (brightness < 0.25 or brightness > 0.75) else "black"
            ax.text(si, mi, txt, ha="center", va="center",
                    fontsize=6, color=color, fontweight="normal")

    cb = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.005, aspect=20)
    cb.ax.tick_params(labelsize=8)

run_info = "  |  ".join(f"{MODELS[m]}: {run_labels[m]}" for m in model_ids if m in run_labels)
fig.suptitle(
    f"Multi-Model Parameter Heatmap — TGPY / Grenada\n{run_info}",
    fontsize=11, y=1.001,
)

plt.savefig("heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: heatmap.png")

In [ ]:
# ── Numerical comparison at key forecast steps ────────────────────────────────
KEY_STEPS  = [24, 48, 72, 96, 120]
KEY_PARAMS = ["t2m_c", "msl_hpa", "wspd_kt", "tp_mm", "tcc_pct", "cape"]
FMT        = {"t2m_c": ".1f", "msl_hpa": ".1f", "wspd_kt": ".1f",
              "tp_mm": ".2f", "tcc_pct": ".0f", "cape": ".0f"}

for step in KEY_STEPS:
    print(f"\nT+{step:03d}h")
    header = f"  {'Model':<18}" + "".join(f"  {PARAMS[p][0][:10]:>10}" for p in KEY_PARAMS)
    print(header)
    print("  " + "─" * (len(header) - 2))
    for mid in model_ids:
        df  = dfs[mid]
        row = df[df["step_h"] == step]
        line = f"  {MODELS[mid]:<18}"
        for pid in KEY_PARAMS:
            if not row.empty and pid in row.columns:
                v = float(row[pid].iloc[0])
                line += f"  {v:{FMT[pid]}:>10}" if not np.isnan(v) else f"  {'—':>10}"
            else:
                line += f"  {'—':>10}"
        print(line)

In [ ]:
# ── Pooled ensemble precipitation probability ─────────────────────────────────
# ECMWF ENS (51) + GEFS (31) = 82-member weighted pool
import cfgrib, xarray as xr

ENS_WEIGHT  = 51 / 82
GEFS_WEIGHT = 31 / 82

def find_latest(model_id, filename):
    root = BASE / model_id / "data"
    if not root.exists():
        return None
    for d in sorted(root.iterdir(), reverse=True):
        p = d / filename
        if p.exists():
            return p
    return None

def nearest_point(ds, lat=12.0042, lon=-61.7862):
    return ds.sel(latitude=lat, longitude=lon, method="nearest", tolerance=0.5)

ens_prob, gefs_prob, windows = {}, {}, []

ens_file = find_latest("ecmwf_ens", "ens_prob_precip_d1-d5.grib2")
if ens_file:
    for ds in cfgrib.open_datasets(str(ens_file)):
        pt = nearest_point(ds)
        for var in ("tpg1", "tpg5"):
            if var in pt.data_vars:
                v = pt[var].values
                ens_prob[var] = v.flatten().tolist() if v.ndim > 0 else [float(v)]
    print(f"ENS:  {ens_file.parent.name}")
else:
    print("ENS:  no data (run ecmwf_ens/download.ipynb)")

gefs_file = find_latest("noaa_gefs", "gefs_prob_precip_d1-d5.nc")
if gefs_file:
    ds_g = xr.open_dataset(gefs_file)
    gefs_prob = {k: ds_g[k].values.tolist() for k in ("tpg1", "tpg5")}
    windows   = ds_g["window"].values.tolist()
    print(f"GEFS: {gefs_file.parent.name}")
else:
    print("GEFS: no data (run noaa_gefs/download.ipynb)")

if not windows:
    windows = [f"{s}-{s+24}h" for s in range(0, 120, 24)]

pooled = {}
for t in ("tpg1", "tpg5"):
    e, g = ens_prob.get(t), gefs_prob.get(t)
    if e and g:
        n = min(len(e), len(g))
        pooled[t] = [ENS_WEIGHT * float(e[i]) + GEFS_WEIGHT * float(g[i]) for i in range(n)]
    elif e:
        pooled[t] = [float(v) for v in e]
    elif g:
        pooled[t] = [float(v) for v in g]

print("\nPooled ensemble precip probability (ENS 51 + GEFS 31 = 82 members)")
print(f"  {'Window':>12}  {'P(>1mm/day)':>12}  {'P(>5mm/day)':>12}  {'Signal':>8}")
print("  " + "─" * 52)
for i, w in enumerate(windows[:5]):
    p1 = pooled.get("tpg1", [None]*5)[i]
    p5 = pooled.get("tpg5", [None]*5)[i]
    p1s = f"{p1:.0%}" if p1 is not None else "—"
    p5s = f"{p5:.0%}" if p5 is not None else "—"
    sig = "HIGH" if (p1 or 0) >= 0.7 else ("MODERATE" if (p1 or 0) >= 0.4 else "LOW")
    print(f"  {str(w):>12}  {p1s:>12}  {p5s:>12}  {sig:>8}")